# 마음숲 학습 1 — 감정 카드 사진 태깅 (trpakov/vit-face-expression)

생성한 **가상 성인 얼굴 사진**에 7감정 태그를 **추천**하고, 사람이 검수해 JSON으로 내려받는다.

| 감정 코드 | anger | disgust | fear | joy | neutral | sad | surprise |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 이름 | 화남 | 혐오 | 두려움 | 기쁨 | 무표정 | 슬픔 | 놀람 |

- 모델 라벨(angry·happy 등)은 위 코드로 바꾼다 (happy → joy, angry → anger). 실행할 때 모델 config의 라벨을 다시 확인한다.
- 넣지 않는 것: 삽화·캐릭터·상황 그림, 실제 아동 사진.
- **모델 추천(aiTag)·생성 의도(intended)·사람 검수(confirmedTag)는 서로 다른 칸이다.** 검수 전 결과를 확정으로 쓰지 않는다.
- **모델 점수로 퀴즈 난이도를 정하지 않는다.** 난이도는 팀·특수교사가 검수 후 정한다.
- 이 모델은 FER2013으로 학습됐다. 여기 결과는 성인 사진에 대한 추천이고 **아동 대상 성능 검증이 아니다.**

순서: 1 설치 → 2 도구 → 3 모델 → 4 사진 올리기 → 5 태깅 → 6 결과 보기 → 7 사람 검수 → 8 JSON 내려받기

## 1. 설치 (런타임: CPU로 충분, GPU면 더 빠름)

In [ ]:
!pip -q install transformers torch pillow opencv-python-headless ipywidgets matplotlib

## 2. 도구 (저장소 `ai/tagging/photo_tagging.py`와 같은 코드)
고치려면 저장소 파일을 고치고 `ai/colab/build_notebook.py`로 노트북을 다시 만든다.

In [ ]:
%%writefile photo_tagging.py
"""학습 1 사진 태깅 (Colab 노트북 ai/colab/photo_tagging.ipynb가 이 파일을 그대로 쓴다).

사진 → 얼굴 1개 검출·자르기 → RGB → trpakov/vit-face-expression → 공통 7감정 점수 → 검토 필요 판정 → JSON.

- 대상: 생성한 가상 성인 얼굴 사진만. 삽화·캐릭터·상황 그림, 실제 아동 사진은 넣지 않는다.
- 모델 점수는 추천일 뿐이다. 사람이 검수한 라벨(confirmedTag)과 구분해 두고, 퀴즈 난이도로 바로 쓰지 않는다.
- 이 모델은 FER2013(흑백 저해상도 얼굴, 주로 성인)으로 학습됐다. 성인 사진에서 나온 결과를 아동 대상 성능으로 보지 않는다.

라벨(모델 config.json의 id2label, 2026-11-28 확인): angry, disgust, fear, happy, neutral, sad, surprise
공통 7감정 코드: anger, disgust, fear, joy, neutral, sad, surprise (happy → joy, angry → anger)
"""
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path

MODEL_ID = "trpakov/vit-face-expression"
EMOTIONS = ("anger", "disgust", "fear", "joy", "neutral", "sad", "surprise")
LABELS_KO = {"anger": "화남", "disgust": "혐오", "fear": "두려움", "joy": "기쁨", "neutral": "무표정",
             "sad": "슬픔", "surprise": "놀람"}
# 모델 라벨 → 공통 코드. 모델이 이 밖의 라벨을 내면 오류로 멈춘다 (check_model_labels).
MODEL_LABELS = {"angry": "anger", "disgust": "disgust", "fear": "fear", "happy": "joy", "neutral": "neutral",
                "sad": "sad", "surprise": "surprise"}
# 파일 이름·메타데이터의 의도 감정으로 받는 다른 이름 (이전 표기 포함)
ALIASES = {"happy": "joy", "angry": "anger", "calm": "neutral", "평온": "neutral", "싫음": "disgust",
           **{v: k for k, v in LABELS_KO.items()}, **{k: k for k in EMOTIONS}}
EXTENSIONS = (".png", ".jpg", ".jpeg", ".webp")
FORMAT = "photo-tags-v1"


@dataclass
class ReviewConfig:
    """검토 필요 기준. 바꾸면 결과 JSON의 config에 남는다."""
    min_top_score: float = 0.5  # 1위 점수가 이보다 낮으면 검토
    require_intended: bool = True  # 의도 감정을 알 수 없으면 검토
    face_margin: float = 0.2  # 자를 때 얼굴 상자 바깥 여백 (상자 크기 대비)
    min_face_px: int = 48  # 이보다 작은 얼굴은 무시


def check_model_labels(id2label: dict) -> dict:
    """모델 config의 id2label이 예상한 7개 라벨인지 확인하고 {모델 라벨: 공통 코드}를 돌려준다."""
    labels = {str(v).lower() for v in id2label.values()}
    if labels != set(MODEL_LABELS):
        raise ValueError(f"모델 라벨이 예상과 달라요: {sorted(labels)} (예상 {sorted(MODEL_LABELS)})")
    return {label: MODEL_LABELS[label] for label in sorted(labels)}


def to_scores(predictions: list[dict]) -> dict:
    """pipeline 출력 [{label, score}] → 공통 7감정 점수 (합이 1이 되게 다시 나눔, 소수 4자리)."""
    scores = {e: 0.0 for e in EMOTIONS}
    for p in predictions:
        label = str(p["label"]).lower()
        if label not in MODEL_LABELS:
            raise ValueError(f"모르는 라벨: {label}")
        scores[MODEL_LABELS[label]] += float(p["score"])
    total = sum(scores.values())
    if total <= 0:
        raise ValueError("점수가 모두 0이에요")
    return {e: round(v / total, 4) for e, v in scores.items()}


def intended_emotion(file_name: str, metadata: dict | None = None) -> tuple[str | None, str | None]:
    """생성할 때 의도한 감정과 그 출처. 메타데이터 intendedEmotion이 먼저, 없으면 파일 이름(person03_sad.png)."""
    if metadata and metadata.get("intendedEmotion"):
        code = ALIASES.get(str(metadata["intendedEmotion"]).strip().lower()) or ALIASES.get(
            str(metadata["intendedEmotion"]).strip())
        return (code, "metadata") if code else (None, None)
    stem = Path(file_name).stem.lower()
    for part in reversed(re.split(r"[_\-\s.]+", stem)):
        if part in ALIASES:
            return ALIASES[part], "filename"
    return None, None


def crop_box(box: tuple[int, int, int, int], width: int, height: int, margin: float) -> tuple[int, int, int, int]:
    """(x, y, w, h) 얼굴 상자에 여백을 더하고 사진 안으로 자른 (left, top, right, bottom)."""
    x, y, w, h = box
    dx, dy = int(w * margin), int(h * margin)
    return max(0, x - dx), max(0, y - dy), min(width, x + w + dx), min(height, y + h + dy)


def review_reasons(record: dict, config: ReviewConfig) -> list[str]:
    """검토 필요 사유. 비어 있으면 검토 필요 없음(그래도 사람 검수 전에는 확정이 아니다)."""
    if record.get("error"):
        return [record["error"]]
    reasons = []
    scores = record["aiScores"]
    top = record["aiTag"]
    if scores[top] < config.min_top_score:
        reasons.append("lowConfidence")
    if record["intended"] is None:
        if config.require_intended:
            reasons.append("noIntended")
    elif record["intended"] != top:
        reasons.append("mismatch")
    return reasons


def empty_review() -> dict:
    """사람 검수 칸. 노트북의 검수 단계에서 채운다. 모델 추천(aiTag)과 따로 둔다."""
    return {"confirmedTag": None, "confirmedBy": None, "confirmedAt": None, "reviewNote": None}


def tag_image(file_name: str, image, detect, classify, config: ReviewConfig, metadata: dict | None = None) -> dict:
    """사진 한 장. image: RGB PIL.Image(또는 같은 모양의 가짜), detect(image) → [(x, y, w, h)...],
    classify(face) → [{label, score}]. 실패는 error 칸에 사유 코드로 남긴다."""
    intended, source = intended_emotion(file_name, metadata)
    record = {"file": file_name, "intended": intended, "intendedSource": source, "faces": None, "faceBox": None,
              "aiTag": None, "aiScores": None, "error": None, **empty_review()}
    try:
        boxes = [b for b in detect(image) if min(b[2], b[3]) >= config.min_face_px]
    except Exception as error:
        record["error"] = "detectError"
        record["errorDetail"] = type(error).__name__
        boxes = None
    if boxes is not None:
        record["faces"] = len(boxes)
        if not boxes:
            record["error"] = "noFace"
        elif len(boxes) > 1:
            record["error"] = "multiFace"
        else:
            box = crop_box(tuple(int(v) for v in boxes[0]), image.width, image.height, config.face_margin)
            record["faceBox"] = list(box)
            try:
                scores = to_scores(classify(image.crop(box).convert("RGB")))
                record["aiScores"] = scores
                record["aiTag"] = max(EMOTIONS, key=lambda e: scores[e])
            except Exception as error:
                record["error"] = "inferenceError"
                record["errorDetail"] = type(error).__name__
    record["reviewReasons"] = review_reasons(record, config)
    record["aiNeedsReview"] = bool(record["reviewReasons"])
    return record


def load_rgb(path: Path):
    from PIL import Image

    with Image.open(path) as img:
        return img.convert("RGB")


def tag_folder(folder: Path, detect, classify, config: ReviewConfig | None = None, load=load_rgb,
               metadata: dict | None = None) -> list[dict]:
    """폴더의 사진 전부. metadata: {파일 이름: {"intendedEmotion": ...}} (선택). 깨진 파일은 fileError."""
    config = config or ReviewConfig()
    records = []
    for path in sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in EXTENSIONS):
        meta = (metadata or {}).get(path.name)
        try:
            image = load(path)
        except Exception as error:
            intended, source = intended_emotion(path.name, meta)
            record = {"file": path.name, "intended": intended, "intendedSource": source, "faces": None,
                      "faceBox": None, "aiTag": None, "aiScores": None, "error": "fileError",
                      "errorDetail": type(error).__name__, **empty_review()}
            record["reviewReasons"] = ["fileError"]
            record["aiNeedsReview"] = True
            records.append(record)
            continue
        records.append(tag_image(path.name, image, detect, classify, config, meta))
    return records


def confirm(record: dict, tag: str, reviewer: str, note: str | None = None, at: str | None = None) -> dict:
    """사람 검수 결과를 기록한다. tag는 공통 7감정 코드 또는 'reject'(쓰지 않음)."""
    if tag not in EMOTIONS and tag != "reject":
        raise ValueError(f"검수 라벨은 7감정 코드 또는 reject: {tag}")
    if not reviewer:
        raise ValueError("검수자 이름이 필요해요")
    return {**record, "confirmedTag": tag, "confirmedBy": reviewer,
            "confirmedAt": at or datetime.now(timezone.utc).isoformat(), "reviewNote": note}


def summarize(records: list[dict]) -> dict:
    n = len(records)
    reasons: dict[str, int] = {}
    for r in records:
        for reason in r["reviewReasons"]:
            reasons[reason] = reasons.get(reason, 0) + 1
    tagged = [r for r in records if r["aiTag"]]
    with_intended = [r for r in tagged if r["intended"]]
    agree = sum(r["aiTag"] == r["intended"] for r in with_intended)
    confirmed = [r for r in records if r["confirmedTag"]]
    return {
        "photos": n, "tagged": len(tagged), "needsReview": sum(r["aiNeedsReview"] for r in records),
        "reasons": reasons,
        # 모델 추천과 생성 의도의 일치율 (정확도가 아니다: 의도한 표정이 잘 나왔는지 사람이 확인해야 한다)
        "agreeWithIntended": {"agree": agree, "of": len(with_intended),
                              "rate": round(agree / len(with_intended), 3) if with_intended else None},
        "confirmed": len(confirmed),
        # 사람 검수 라벨 기준 모델 일치율 (검수가 끝난 사진만)
        "agreeWithConfirmed": {
            "agree": sum(r["aiTag"] == r["confirmedTag"] for r in confirmed if r["aiTag"]),
            "of": sum(1 for r in confirmed if r["aiTag"] and r["confirmedTag"] != "reject")},
    }


def build_document(records: list[dict], config: ReviewConfig, model_info: dict, created_at: str | None = None) -> dict:
    """결과 JSON. difficulty는 정하지 않는다 (모델 점수를 퀴즈 난이도로 바로 쓰지 않음)."""
    return {
        "format": FORMAT,
        "createdAt": created_at or datetime.now(timezone.utc).isoformat(),
        "model": model_info,
        "labelMap": MODEL_LABELS,
        "emotions": {e: LABELS_KO[e] for e in EMOTIONS},
        "config": asdict(config),
        "scope": "가상 성인 얼굴 사진의 태그 추천. 아동 대상 성능 검증이 아니다.",
        "difficultyPolicy": "모델 점수로 퀴즈 난이도를 정하지 않는다. 난이도는 팀·특수교사가 검수 후 따로 정한다.",
        "summary": summarize(records),
        "records": records,
    }


def save(doc: dict, path: Path) -> Path:
    path = Path(path)
    path.write_text(json.dumps(doc, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    return path


# ---------- Colab에서 실제로 쓰는 검출기·분류기 (설치가 필요해 함수 안에서 불러온다) ----------
def opencv_detector(scale_factor: float = 1.1, min_neighbors: int = 5):
    """OpenCV Haar 정면 얼굴 검출기 (Colab에 기본 설치). 옆얼굴·가림에는 약하다 → 얼굴 없음은 검토로 남는다."""
    import cv2
    import numpy as np

    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    def detect(image):
        gray = cv2.cvtColor(np.asarray(image.convert("RGB")), cv2.COLOR_RGB2GRAY)
        found = cascade.detectMultiScale(gray, scaleFactor=scale_factor, minNeighbors=min_neighbors)
        return [tuple(int(v) for v in b) for b in found]

    return detect


def hf_classifier(model_id: str = MODEL_ID, revision: str | None = None, device: int | None = None):
    """transformers 이미지 분류 파이프라인. 모델 라벨을 확인하고 (classify, model_info)를 돌려준다."""
    from transformers import pipeline

    pipe = pipeline("image-classification", model=model_id, revision=revision, device=device)
    label_map = check_model_labels(pipe.model.config.id2label)
    info = {"id": model_id, "revision": revision or getattr(pipe.model.config, "_commit_hash", None),
            "labels": sorted(label_map)}

    def classify(face):
        return pipe(face, top_k=len(MODEL_LABELS))

    return classify, info

In [ ]:
import importlib, photo_tagging as pt
importlib.reload(pt)
config = pt.ReviewConfig(min_top_score=0.5, require_intended=True)  # 검토 기준 (바꾸면 결과 JSON에 남음)
config

## 3. 모델 불러오기
라벨이 예상과 다르면 여기서 멈춘다.

In [ ]:
classify, model_info = pt.hf_classifier()  # 특정 버전으로 고정하려면 revision="커밋 해시"
detect = pt.opencv_detector()
model_info

## 4. 사진 올리기
- 파일 이름에 의도한 감정을 넣는다: `person03_sad.png`, `p01_joy.webp` (이전 표기 happy·angry·평온·싫음도 읽음)
- 또는 메타데이터 JSON `{"파일이름.png": {"intendedEmotion": "sad"}, ...}`을 함께 올린다 (파일 이름보다 우선)
- Google Drive 폴더를 쓰려면 아래 `PHOTO_DIR`을 Drive 경로로 바꾸고 업로드 셀은 건너뛴다

In [ ]:
from pathlib import Path
import json
PHOTO_DIR = Path('/content/photos')
PHOTO_DIR.mkdir(parents=True, exist_ok=True)
metadata = {}
try:
    from google.colab import files
    for name, data in files.upload().items():
        if name.endswith('.json'):
            metadata.update(json.loads(data.decode('utf-8')))
        else:
            (PHOTO_DIR / name).write_bytes(data)
except ImportError:
    print('Colab이 아니에요. PHOTO_DIR에 사진을 직접 넣어 주세요.')
sorted(p.name for p in PHOTO_DIR.iterdir())

## 5. 태깅 (얼굴 1개 검출 → 자르기 → RGB → 7감정 점수 → 검토 사유)

In [ ]:
records = pt.tag_folder(PHOTO_DIR, detect, classify, config, metadata=metadata)
pt.summarize(records)

## 6. 결과 보기
왼쪽 원본(검출 상자), 가운데 잘라낸 얼굴, 오른쪽 7감정 점수. 제목에 의도·추천·검토 사유.
사유: noFace(얼굴 없음) · multiFace(여러 얼굴) · fileError · detectError · inferenceError ·
lowConfidence(1위 < 기준) · mismatch(추천 ≠ 의도) · noIntended(의도를 모름)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import patches
plt.rcParams['font.family'] = 'DejaVu Sans'  # 한글 글꼴이 없으면 코드로 표시
for r in records:
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    try:
        img = pt.load_rgb(PHOTO_DIR / r['file'])
        axes[0].imshow(img)
        if r['faceBox']:
            l, t, rr, b = r['faceBox']
            axes[0].add_patch(patches.Rectangle((l, t), rr - l, b - t, fill=False, color='lime', lw=2))
            axes[1].imshow(img.crop(tuple(r['faceBox'])))
    except Exception as e:
        axes[0].text(0.1, 0.5, 'cannot open: ' + type(e).__name__)
    if r['aiScores']:
        axes[2].barh(list(r['aiScores']), list(r['aiScores'].values()), color=['tab:green' if k == r['aiTag'] else 'tab:gray' for k in r['aiScores']])
        axes[2].set_xlim(0, 1)
    for ax in axes[:2]:
        ax.axis('off')
    fig.suptitle(f"{r['file']} | intended={r['intended']} | ai={r['aiTag']} | review={','.join(r['reviewReasons']) or '-'}")
    plt.show()

## 7. 사람 검수
사진마다 검수 라벨(7감정 코드 또는 reject=쓰지 않음)을 고르고, 검수자 이름을 적은 뒤 **검수 저장**을 누른다.
'미검수'로 둔 사진은 confirmedTag가 비어 있고, 앱 콘텐츠에 넣지 않는다.

In [ ]:
import ipywidgets as w
from IPython.display import display
reviewer = w.Text(description='검수자', placeholder='이름')
choices = [('미검수', None)] + [(f'{e} {pt.LABELS_KO[e]}', e) for e in pt.EMOTIONS] + [('reject 쓰지 않음', 'reject')]
pickers = []
for r in records:
    default = r['confirmedTag']
    pick = w.Dropdown(options=choices, value=default, description=r['file'][:18], layout=w.Layout(width='420px'))
    note = w.Text(placeholder='메모 (선택)', layout=w.Layout(width='300px'))
    pickers.append((r, pick, note))
    display(w.HBox([pick, note, w.HTML(f"추천 <b>{r['aiTag']}</b>, 의도 {r['intended']}, 사유 {','.join(r['reviewReasons']) or '-'}")]))
out = w.Output()
def apply(_):
    out.clear_output()
    with out:
        if not reviewer.value.strip():
            print('검수자 이름을 적어 주세요.'); return
        done = 0
        for i, (r, pick, note) in enumerate(pickers):  # pickers는 records와 같은 순서
            if pick.value:
                records[i] = pt.confirm(r, pick.value, reviewer.value.strip(), note.value or None)
                pickers[i] = (records[i], pick, note)
                done += 1
        print(f'검수 저장: {done}장')
button = w.Button(description='검수 저장', button_style='success')
button.on_click(apply)
display(reviewer, button, out)

## 8. JSON 저장·내려받기

In [ ]:
from datetime import datetime
doc = pt.build_document(records, config, model_info)
path = pt.save(doc, Path(f"/content/photo_tags_{datetime.now():%Y%m%d_%H%M}.json"))
print(doc['summary'])
try:
    from google.colab import files
    files.download(str(path))
except ImportError:
    print('저장:', path)

## 전달 절차
1. 내려받은 `photo_tags_*.json`을 AI 담당에게 전달한다 (사진 원본은 팀 드라이브, 저장소에 넣지 않음).
2. AI 담당은 `docs/ai/tagging/`에 결과를 두고, **confirmedTag가 있고 reject가 아닌 사진만** 콘텐츠 후보로 쓴다.
3. 앱에 넣을 때는 `docs/ai/HANDOFF-AI.md` '학습 1 — 앱 콘텐츠 반영'을 따르고 `frontend/public/art/manifest.json`에 기록한다.
4. 요약의 agreeWithIntended(추천과 의도의 일치)는 정확도가 아니다. 사람 검수 후 agreeWithConfirmed를 본다.